In [1]:
from pathlib import Path

from dustmaps.config import config as dustmaps_config
import dustmaps.sfd
from dustmaps.sfd import SFDQuery
from astropy import table
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from plotters import (
    plot_aitoff_reddening,
    plot_aitoff_sfd,
    plot_sfd_regime_decomposition,
)

The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [ ]:
data_path = Path("rrlyrae_reddening_sample.npz")

data = np.load(data_path, allow_pickle=True)
rrlyrae_clean = table.Table({k: data[k] for k in data.files})

l_deg = rrlyrae_clean["l"]
b_deg = rrlyrae_clean["b"]

fig, ax = plot_aitoff_reddening(l_deg, b_deg, rrlyrae_clean["E_bprp"])
plt.show()

In [ ]:
DUSTMAPS_DATA_DIR = Path(".dustmaps-data")
DUSTMAPS_DATA_DIR.mkdir(parents=True, exist_ok=True)
dustmaps_config["data_dir"] = str(DUSTMAPS_DATA_DIR.resolve())

try:
    sfd = SFDQuery()
except FileNotFoundError:
    dustmaps.sfd.fetch()
    sfd = SFDQuery()

coords = SkyCoord(l=l_deg * u.deg, b=b_deg * u.deg, frame="galactic")
sfd_ebv = np.array(sfd(coords), dtype=float)

fig, ax = plot_aitoff_sfd(l_deg, b_deg, sfd_ebv)
plt.show()

In [ ]:
SIMILAR_SCALE_MAX = 2.0
LARGE_SFD_MIN = 10.0

empirical = rrlyrae_clean["E_bprp"]

finite_mask = np.isfinite(sfd_ebv) & np.isfinite(empirical)
similar_mask = finite_mask & (sfd_ebv <= SIMILAR_SCALE_MAX)
large_mask = finite_mask & (sfd_ebv > LARGE_SFD_MIN)

x_sim = sfd_ebv[similar_mask]
y_sim = empirical[similar_mask]
slope, intercept = np.polyfit(x_sim, y_sim, 1)

axes = plot_sfd_regime_decomposition(
    sfd_ebv, empirical, similar_mask, large_mask,
    slope, intercept,
    similar_scale_max=SIMILAR_SCALE_MAX, large_sfd_min=LARGE_SFD_MIN,
)
plt.show()